In [3]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import wandb

PALETTE = {
    "lejepa":  "#2D68C4", 
    "gsvit": "#50C878", 
}
ACCENT = "#F3F2F2"
GRID = "#C1C1C1"
FONT = "Inter, sans-serif"
AXIS_STYLE = dict(
    showline=False, linecolor="black", linewidth=1, mirror=False,
    ticks="outside", tickcolor="black", ticklen=4,
    showgrid=True, gridcolor=GRID, gridwidth=1, griddash="dot",
    zeroline=False,
)

## Helpers

In [4]:
api = wandb.Api()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/sinal/.netrc.


In [5]:
def fetch(run_path, keys):
    run = api.run(run_path)
    rows = list(run.scan_history(keys=keys))   # full resolution, no 500-pt cap
    return pd.DataFrame(rows)

In [6]:
def fetch_table(run_path, log_key, artifact_name=None):
    """log_key = the key passed to wandb.log({log_key: table}).
       artifact_name = optional override (use if wandb appended a suffix)."""
    run = api.run(run_path)
    if artifact_name is None:
        artifact_name = f"run-{run.id}-{log_key.replace('/', '')}:latest"
    artifact = api.artifact(f"{run.entity}/{run.project}/{artifact_name}")
    table = artifact.get(log_key)
    if table is None:
        files = [f.name for f in artifact.files()]
        raise KeyError(f"No table at {log_key!r}. Files: {files}")
    return pd.DataFrame(table.data, columns=table.columns)

In [7]:
def log_artifacts(run_path):
    """Get all artifacts for the run"""
    run = api.run(run_path)
    for art in run.logged_artifacts():
        print(art.name, art.type)


In [8]:
def styled_fig(title, xlabel, ylabel, width=620, height=400):
    fig = go.Figure()
    fig.update_layout(
        title=dict(text=f"<b>{title}</b>", x=0.5, xanchor="center",
                   font=dict(family=FONT, size=15, color="black")),
        width=width, height=height,
        margin=dict(l=70, r=30, t=60, b=60),
        plot_bgcolor=ACCENT, paper_bgcolor=ACCENT,
        font=dict(family=FONT, size=13, color="black"),
    )
    fig.update_xaxes(title_text=f"<b>{xlabel}</b>", **AXIS_STYLE)
    fig.update_yaxes(title_text=f"<b>{ylabel}</b>", **AXIS_STYLE)
    return fig

## Pretraining results

In [6]:
lejepa_df = fetch(
    "mrsinal-tilburg-university/jepa/runs/0vy2l943",
    keys=["trainer/global_step", "epoch", "fit/loss", "fit/pred_loss", "fit/sigreg_loss", "_runtime"])

In [7]:
fig = styled_fig("LeJEPA Training Loss", "Step", "Training Loss")
fig.add_scatter(
    x=lejepa_df["trainer/global_step"],
    y=lejepa_df["fit/loss"],
    mode="lines",
    line=dict(color=PALETTE["lejepa"], width=2),
    name="LeJEPA",
)
fig.show()
fig.write_image("assests/lejepa_train_loss.svg")  # needs `kaleido`

In [8]:
gsvit_df = fetch(
    "mrsinal-tilburg-university/gsvit/runs/qarvfhso",
    keys=["_step", "epoch", "loss", "running_loss_lg", "running_loss", "_runtime"])

In [9]:
fig = styled_fig("GSViT Training Loss", "Step", "Training Loss")
fig.add_scatter(
    x=gsvit_df["_step"],
    y=gsvit_df["loss"],
    mode="lines",
    line=dict(color=PALETTE["gsvit"], width=2),
    name="GSViT",
)
fig.show()
fig.write_image("assests/gsvit_train_loss.svg")  # needs `kaleido`

In [10]:
gpu_days = [max(lejepa_df["_runtime"]) / 86400 , max(gsvit_df["_runtime"]) / 86400]

In [11]:
fig = styled_fig("Training Cost", "Models", "Training GPU-Days")
fig.add_bar(
    x=["LeJEPA", "GSViT"], y=gpu_days,
    marker=dict(
        color=[PALETTE["lejepa"], PALETTE["gsvit"]],
        line=dict(color=ACCENT, width=1),
    ),
    showlegend=False,
)
fig.show()
fig.write_image("assests/training_costs.svg")  # needs `kaleido`

## Layer-wise probe

In [23]:
print(log_artifacts("mrsinal-tilburg-university/gsvit/runs/h0zr59yy"))

run-h0zr59yy-proberesults_table-Molrxg:v0 run_table


In [9]:
gsvit_probe_df = fetch_table(
    "mrsinal-tilburg-university/gsvit/runs/h0zr59yy",
    log_key="probe/results_table",                       # original key, with slash
    artifact_name="run-h0zr59yy-proberesults_table-Molrxg:latest")

wandb:   1 of 1 files downloaded.  


In [30]:
gsvit_probe_df.head()

,layer,accuracy_mean,accuracy_std,f1_mean,f1_std,jaccard_mean,jaccard_std,best_lr_per_fold,best_wd_per_fold
0,0,0.518806,0.025638,0.241465,0.033887,0.161214,0.021905,0.003;0.001;0.005;0.005;0.003,0.01;0.8;0.1;0.4;0.8
1,1,0.491614,0.022567,0.203081,0.021253,0.134873,0.013275,0.003;0.005;0.001;0.0003;0.003,0.4;0.01;0.01;0.8;0.01
2,2,0.513350,0.023209,0.207637,0.032951,0.140833,0.019445,0.005;0.003;0.003;0.005;0.003,0.4;0.1;0.01;0.1;0.8
3,3,0.516019,0.022764,0.228258,0.018723,0.153335,0.013653,0.001;0.003;0.003;0.003;0.003,0.8;0.8;0.01;0.1;0.8
4,4,0.515164,0.021677,0.220720,0.024601,0.148616,0.016305,0.003;0.001;0.0003;0.001;0.003,0.4;0.1;0.8;0.1;0.8


In [50]:
fig = styled_fig("Mean of Probe Metrics (GSViT)", "Layer", "Score")
fig.update_xaxes(
    tickmode="array",
    tickvals=gsvit_probe_df["layer"].tolist(), 
)

for col, name, color in [
    ("accuracy_mean", "Accuracy", PALETTE["gsvit"]),
    ("f1_mean",       "F1",       PALETTE["lejepa"]),
    ("jaccard_mean",  "Jaccard",  "black"),
]:
    fig.add_scatter(
        x=gsvit_probe_df["layer"], y=gsvit_probe_df[col],
        mode="lines+markers", name=name,
        line=dict(color=color, width=2),
    )

fig.show()
fig.write_image("assests/gsvit_probe_mean.svg")  # needs `kaleido`

In [51]:
fig = styled_fig("Standard deviation of Probe Metrics (GSViT)", "Layer", "Score")
fig.update_xaxes(
    tickmode="array",
    tickvals=gsvit_probe_df["layer"].tolist(), 
)

for col, name, color in [
    ("accuracy_std", "Accuracy", PALETTE["gsvit"]),
    ("f1_std",       "F1",       PALETTE["lejepa"]),
    ("jaccard_std",  "Jaccard",  "black"),
]:
    fig.add_scatter(
        x=gsvit_probe_df["layer"], y=gsvit_probe_df[col],
        mode="lines+markers", name=name,
        line=dict(color=color, width=2),
    )

fig.show()
fig.write_image("assests/gsvit_probe_std.svg")  # needs `kaleido`

In [52]:
print(log_artifacts("mrsinal-tilburg-university/jepa/runs/3yhpy930"))

run-3yhpy930-proberesults_table-5EgRBw:v0 run_table
None


In [12]:
jepa_probe_df = fetch_table(
    "mrsinal-tilburg-university/jepa/runs/3yhpy930",
    log_key="probe/results_table",                       # original key, with slash
    artifact_name="run-3yhpy930-proberesults_table-5EgRBw:latest")

wandb:   1 of 1 files downloaded.  


In [55]:
fig = styled_fig("Mean of Probe Metrics (LeJEPA)", "Layer", "Score")
fig.update_xaxes(
    tickmode="array",
    tickvals=jepa_probe_df["layer"].tolist(), 
)

for col, name, color in [
    ("accuracy_mean", "Accuracy", PALETTE["gsvit"]),
    ("f1_mean",       "F1",       PALETTE["lejepa"]),
    ("jaccard_mean",  "Jaccard",  "black"),
]:
    fig.add_scatter(
        x=jepa_probe_df["layer"], y=jepa_probe_df[col],
        mode="lines+markers", name=name,
        line=dict(color=color, width=2),
    )

fig.show()
fig.write_image("assests/jepa_probe_mean.svg")  # needs `kaleido`

In [56]:
fig = styled_fig("Standard deviation of Probe Metrics (LeJEPA)", "Layer", "Score")
fig.update_xaxes(
    tickmode="array",
    tickvals=jepa_probe_df["layer"].tolist(), 
)

for col, name, color in [
    ("accuracy_std", "Accuracy", PALETTE["gsvit"]),
    ("f1_std",       "F1",       PALETTE["lejepa"]),
    ("jaccard_std",  "Jaccard",  "black"),
]:
    fig.add_scatter(
        x=jepa_probe_df["layer"], y=jepa_probe_df[col],
        mode="lines+markers", name=name,
        line=dict(color=color, width=2),
    )

fig.show()
fig.write_image("assests/jepa_probe_std.svg")  # needs `kaleido`

In [69]:
def summarize(df, label):
    return {
        "model": label,
        "best_layer": int(df["accuracy_mean"].idxmax()),
        "best_acc":  df["accuracy_mean"].max(),
        "best_f1":   df["f1_mean"].max(),
        "best_jac":  df["jaccard_mean"].max(),
        "final_acc": df["accuracy_mean"].iloc[-1],
        "final_f1":  df["f1_mean"].iloc[-1],
        "final_jac": df["jaccard_mean"].iloc[-1],
    }

pd.DataFrame([summarize(jepa_probe_df, "LeJEPA"),
              summarize(gsvit_probe_df, "GSViT")])


,model,best_layer,best_acc,best_f1,best_jac,final_acc,final_f1,final_jac
0,LeJEPA,7,0.556029,0.321414,0.213122,0.523762,0.251348,0.167027
1,GSViT,0,0.518806,0.241465,0.161214,0.473932,0.167222,0.111933


In [19]:
df = pd.concat(
    [gsvit_probe_df.assign(model="GSViT"), jepa_probe_df.assign(model="LeJEPA")],
    ignore_index=True,
)

# Pick the columns you want, set the MultiIndex
metrics = ["accuracy_mean", "accuracy_std", "jaccard_mean", "jaccard_std"]
layered_table = df.set_index(["model", "layer"])[metrics]


In [20]:
layered_table.columns = pd.MultiIndex.from_tuples(
    [tuple(c.rsplit("_", 1)) for c in layered_table.columns],
    names=["metric", "stat"],
)

In [22]:
def fmt(m, s):
    return f"{m:.3f} ± {s:.3f}"

pretty = pd.DataFrame({
    "Accuracy": [fmt(m, s) for m, s in zip(df.accuracy_mean, df.accuracy_std)],
    "F1":       [fmt(m, s) for m, s in zip(df.f1_mean, df.f1_std)],
    "Jaccard":  [fmt(m, s) for m, s in zip(df.jaccard_mean, df.jaccard_std)],
}, index=pd.MultiIndex.from_frame(df[["model", "layer"]]))


In [ ]:
print(pretty.to_latex(multirow=True, column_format="llccc"))

'\\begin{tabular}{llccc}\n\\toprule\n &  & Accuracy & F1 & Jaccard \\\\\nmodel & layer &  &  &  \\\\\n\\midrule\n\\multirow[t]{13}{*}{GSViT} & 0 & 0.519 ± 0.026 & 0.241 ± 0.034 & 0.161 ± 0.022 \\\\\n & 1 & 0.492 ± 0.023 & 0.203 ± 0.021 & 0.135 ± 0.013 \\\\\n & 2 & 0.513 ± 0.023 & 0.208 ± 0.033 & 0.141 ± 0.019 \\\\\n & 3 & 0.516 ± 0.023 & 0.228 ± 0.019 & 0.153 ± 0.014 \\\\\n & 4 & 0.515 ± 0.022 & 0.221 ± 0.025 & 0.149 ± 0.016 \\\\\n & 5 & 0.507 ± 0.019 & 0.216 ± 0.015 & 0.144 ± 0.010 \\\\\n & 6 & 0.494 ± 0.028 & 0.213 ± 0.028 & 0.140 ± 0.019 \\\\\n & 7 & 0.502 ± 0.019 & 0.181 ± 0.026 & 0.124 ± 0.015 \\\\\n & 8 & 0.492 ± 0.015 & 0.206 ± 0.007 & 0.136 ± 0.005 \\\\\n & 9 & 0.485 ± 0.014 & 0.191 ± 0.047 & 0.127 ± 0.026 \\\\\n & 10 & 0.481 ± 0.020 & 0.197 ± 0.039 & 0.130 ± 0.022 \\\\\n & 11 & 0.471 ± 0.033 & 0.203 ± 0.037 & 0.132 ± 0.021 \\\\\n & 12 & 0.474 ± 0.025 & 0.167 ± 0.037 & 0.112 ± 0.023 \\\\\n\\cline{1-5}\n\\multirow[t]{13}{*}{LeJEPA} & 0 & 0.502 ± 0.031 & 0.169 ± 0.015 & 0.118 ± 0